# exp084 — kernel exp080-ctrl224-fast más rápido con las mismas predicciones (bench Kaggle T4×2)

Padre: exp080_seqhead_ctrl_mil (kernel `exp080-ctrl224-fast`, LB 0.949). Experimento de PRODUCTO / EFICIENCIA: X1 prep
temprana (pool antes de importar torch), X2 recorte 320 en los procesos de prep, X3 un hilo cv2 por proceso, X4 ventanas
GPU desde el recorte, X5 cudnn.benchmark. No se entrena nada. Los bench se generan con `gen_bench3.py` (+ `fastx.py`) de
esta carpeta, corren como kernels privados de Kaggle y sus salidas (`bench_report.json`, `raw_probs.csv`) se copian a
`results/exp084_kernel_rapido/bench/<kernel>_v<n>/` (fuera de git salvo metrics.json). Ver SPEC.md.

Celdas fijas: **0** parámetros · **1** setup · **2** datos (salidas de los bench) · **3** "modelo" (transformación) ·
**4** sin entrenamiento · **5** evaluación · **6** guardar · **7** verdict.


## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp084_kernel_rapido"
PADRE = "exp080"
SMOKE = False
SELECT_ON = "pseudo"           # no aplica: no hay selección (bench de runtime)
BENCH = "bench"                # results/<EXP>/bench/<kernel>_v<n>/{bench_report.json, raw_probs.csv}
BASELINE_RAW = "exp080-bench-fastng6_v1"          # raw_probs.csv de referencia (eff2, 17-oct)
PARENT_RUNS = ["exp080-bench-fastng6_v1", "exp080-bench-fastng6_v2"]
DECIDE_RUNS = ['exp084-bench-xb_v1']   # corridas de la variante que decide
INFO_RUNS = ['exp084-bench-x_v1']     # informativas
CRIT_DPROB = 1e-3              # SPEC: max|Δ prob| en los 1300 estudios
CRIT_DT = -30.0                # SPEC: media nb_elapsed_to_end (decide) − media padre <= −30 s
NOTES = "exp084: runtime con predicciones iguales; tiempo = nb_elapsed_to_end del bench (incluye imports, hash, carga y bucle)"


## 1. Setup

In [ ]:
import json, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.experiments import git_info, guardar_metricas
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
GIT = git_info(REPO)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']})")


## 2. Datos: salidas de los bench (las revisiones detienen el notebook si algo está mal)

In [ ]:
def load_run(name):
    d = RESULTS / BENCH / name
    rep = json.load(open(d / "bench_report.json"))
    raw = pd.read_csv(d / "raw_probs.csv"); raw["StudyInstanceUID"] = raw.StudyInstanceUID.astype(str)
    logs = sorted(d.glob("*.log"))
    assert logs, f"sin log en {d}"
    log = logs[0].read_text(errors="ignore")
    return rep, raw.set_index("StudyInstanceUID"), log

RUNS = {n: load_run(n) for n in dict.fromkeys(PARENT_RUNS + DECIDE_RUNS + INFO_RUNS + [BASELINE_RAW])}
REF = RUNS[BASELINE_RAW][1]
assert len(REF) == 1300
for n, (rep, raw, log) in RUNS.items():
    assert rep["n"] == 1300 and set(raw.index) == set(REF.index), n
print({n: r[0]["variant"] for n, r in RUNS.items()})


## 3. Transformación (fastx.py): ver SPEC; aquí solo se comprueba que los bench la contienen

In [ ]:
import ast
FASTX = (HERE / "fastx.py").read_text() if (HERE / "fastx.py").exists() else (REPO / "experiments" / EXP / "fastx.py").read_text()
ast.parse(FASTX)
print("fastx.py:", len(FASTX.splitlines()), "líneas; cambios X1-X5 documentados en su docstring")


## 4. Sin entrenamiento: tabla por corrida

In [ ]:
rows = {}
for n, (rep, raw, log) in RUNS.items():
    d = np.abs(raw.loc[REF.index, LABELS].to_numpy() - REF[LABELS].to_numpy())
    t = rep["times"]
    rows[n] = {"variant": rep["variant"], "nb_elapsed_to_end": rep["nb_elapsed_to_end"], "main_total_s": t.get("main_total_s"),
               "pre_done": t.get("pre_done"), "gpu_done": t.get("gpu_done"), "early_start_s": rep.get("early_start_s"),
               "gpu_wait_sum": float(sum(rep["gpu_wait"].values())), "gpu_mean": rep["gpu_mean"],
               "pre_wall_mean": rep["pre_wall_mean"], "pre_cpu_mean": rep["pre_cpu_mean"], "gold_auc": rep["gold_auc"],
               "max_abs_dprob": float(d.max()), "mean_abs_dprob": float(d.mean()),
               "traceback": ("Traceback" in log) or ("FALLBACK" in log)}
TABLE = pd.DataFrame(rows).T
display(TABLE)


## 5. Evaluación (criterios de la SPEC)

In [ ]:
par = TABLE.loc[PARENT_RUNS, "nb_elapsed_to_end"].astype(float)
dec = TABLE.loc[DECIDE_RUNS]
inf = TABLE.loc[INFO_RUNS]
DT = float(dec["nb_elapsed_to_end"].astype(float).mean() - par.mean())
DT_INFO = float(inf["nb_elapsed_to_end"].astype(float).mean() - par.mean())
OK_PROB = bool((dec["max_abs_dprob"].astype(float) <= CRIT_DPROB).all())
OK_CLEAN = bool(not dec["traceback"].astype(bool).any())
if not (OK_PROB and OK_CLEAN):
    VERDICT_AUTO = "killed"
elif DT <= CRIT_DT:
    VERDICT_AUTO = "worked"
else:
    VERDICT_AUTO = "didnt"
INFO_OK = bool((inf["max_abs_dprob"].astype(float) <= CRIT_DPROB).all() and DT_INFO <= CRIT_DT)
print(f"padre {par.round(1).tolist()} (media {par.mean():.1f}) · decide {dec['nb_elapsed_to_end'].astype(float).round(1).tolist()} "
      f"-> Δ {DT:+.1f} s · max|Δp| {dec['max_abs_dprob'].astype(float).max():.2e} · limpio {OK_CLEAN} -> VERDICT_AUTO {VERDICT_AUTO}")
print(f"informativo x: Δ {DT_INFO:+.1f} s · max|Δp| {inf['max_abs_dprob'].astype(float).max():.2e} · cumple {INFO_OK}")


## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE, "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(), "phase": "bench",
    "select_on": "no aplica (bench de runtime, predicciones iguales)",
    "params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))},
    "runs": {n: {k: (float(v) if isinstance(v, (np.floating, float, int)) and not isinstance(v, bool) else v)
                 for k, v in r.items()} for n, r in rows.items()},
    "parent_mean_nb_elapsed": float(par.mean()), "decide_mean_nb_elapsed": float(dec["nb_elapsed_to_end"].astype(float).mean()),
    "delta_s": DT, "delta_s_info_x": DT_INFO, "max_abs_dprob_decide": float(dec["max_abs_dprob"].astype(float).max()),
    "ok_prob": OK_PROB, "ok_clean": OK_CLEAN, "info_x_meets": INFO_OK,
    "gold_auc_bench": {n: float(r["gold_auc"]) for n, r in rows.items()},
    "verdict_auto": VERDICT_AUTO,
    "criteria": {"worked": f"max|dprob| <= {CRIT_DPROB} (2 runs), no Traceback/FALLBACK, mean nb_elapsed_to_end - parent <= {CRIT_DT} s",
                 "killed": "dprob or clean fails"},
    "gpu_hours_total": 0.0,
}
print("Guardado:", guardar_metricas(RESULTS, metrics))


## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza la fila en `docs/EXPERIMENTOS.md`.
